# Fabric Data Agent - Factory

This notebook automatically scans all workspaces in a Fabric tenant, profiles data items
(Lakehouses, Warehouses, Semantic Models, KQL Databases), uses Azure OpenAI to cluster
them into thematic domains, and creates fully configured Data Agents with system instructions,
data instructions, and example queries.

**Phases:**
1. Workspace & Item Discovery
2. Schema & Data Profiling
3. LLM-based Topic Clustering
4. Data Agent Creation & Configuration
5. Reporting & Agent Garden Catalog

## Cell 0 — Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Edit these values before running
# ============================================================

# Azure OpenAI settings (for topic clustering)
AOAI_ENDPOINT = "https://FoundryFabric.cognitiveservices.azure.com/"   # Azure AI Foundry endpoint
deployment_name = "gpt-4.1-mini"     
resource_name = "FoundryFabric"

# Service principal config for accessing Foundry APIs (required to read catalog and create agents)
# Set these as environment variables or use Key Vault — never hardcode secrets!
import os
tenant_id = os.environ["FABRIC_TENANT_ID"]
client_id = os.environ["FABRIC_CLIENT_ID"]
client_secret = os.environ["FABRIC_CLIENT_SECRET"]

# Profiling settings
SAMPLE_ROWS = 5              # Number of sample rows to read per table
MAX_TABLES_PER_ITEM = 50     # Skip items with more tables than this (very large schemas)
MAX_COLUMNS_IN_SAMPLE = 20   # Limit columns sent to LLM to control token usage

# Agent creation settings
AGENT_NAME_PREFIX = "Auto"                          # Prefix for auto-created agent names
AGENT_TARGET_WORKSPACE = None                       # Workspace name to create agents in (None = current notebook workspace)
MAX_DATASOURCES_PER_AGENT = 5                       # SDK limit
MAX_EXAMPLES_PER_DATASOURCE = 20                    # How many example queries to generate (max 100)

# Discovery settings
WORKSPACE_FILTER = None        # Set to a list of workspace names to limit scanning, e.g. ["Sales WS", "Finance WS"]. None = all.
ITEM_TYPES_TO_SCAN = ["Lakehouse", "Warehouse", "SemanticModel", "KQLDatabase"]

# Catalog output
CATALOG_LAKEHOUSE = None       # Lakehouse name to write the agent catalog to (None = skip catalog write)
CATALOG_TABLE_NAME = "agent_garden_catalog"

# Safety
DRY_RUN = True                 # If True, only plan agents but don't create them. Set to False to actually create.

print("Configuration loaded.")
print(f"  DRY_RUN = {DRY_RUN} — {'Will NOT create agents (preview only)' if DRY_RUN else 'Will CREATE and PUBLISH agents'}")

## Cell 1 — Install Dependencies

In [ ]:
%pip install fabric-data-agent-sdk openai --quiet

In [ ]:
import json
import time
import re
from collections import defaultdict
from typing import Any

import pandas as pd
import sempy.fabric as fabric
from openai import AzureOpenAI
from synapse.ml.mlflow import get_mlflow_env_config

from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent,
)

print("All imports successful.")

---
## Phase 1 — Workspace & Item Discovery

Enumerates all accessible workspaces and lists data items (Lakehouses, Warehouses, Semantic Models, KQL Databases).

In [ ]:
def discover_items(workspace_filter: list[str] | None, item_types: list[str]) -> pd.DataFrame:
    """Discover all data items across accessible workspaces."""
    
    # List all workspaces the current user can access
    workspaces_df = fabric.list_workspaces()
    print(f"Found {len(workspaces_df)} accessible workspaces.")
    
    if workspace_filter:
        workspaces_df = workspaces_df[workspaces_df["Name"].isin(workspace_filter)]
        print(f"Filtered to {len(workspaces_df)} workspaces matching filter.")
    
    all_items = []
    
    for _, ws in workspaces_df.iterrows():
        ws_id = ws["Id"]
        ws_name = ws["Name"]
        
        try:
            items_df = fabric.list_items(workspace=ws_id)
        except Exception as e:
            print(f"  [WARN] Cannot list items in '{ws_name}': {e}")
            continue
        
        # Filter for data item types
        data_items = items_df[items_df["Type"].isin(item_types)]
        
        for _, item in data_items.iterrows():
            all_items.append({
                "workspace_id": ws_id,
                "workspace_name": ws_name,
                "item_id": item["Id"],
                "item_name": item["Display Name"],
                "item_type": item["Type"],
            })
        
        if len(data_items) > 0:
            print(f"  [{ws_name}] Found {len(data_items)} data items")
    
    inventory_df = pd.DataFrame(all_items)
    print(f"\nTotal: {len(inventory_df)} data items across {inventory_df['workspace_name'].nunique()} workspaces.")
    return inventory_df


# Run discovery
item_inventory_df = discover_items(WORKSPACE_FILTER, ITEM_TYPES_TO_SCAN)
display(item_inventory_df)

---
## Phase 2 — Schema & Data Profiling

For each discovered data item, reads the schema (tables + columns) and samples a few rows to understand the data context.

In [ ]:
# ----- Profiling helpers per item type -----

def profile_lakehouse_or_warehouse(item_name: str, item_id: str, item_type: str, workspace_id: str, sample_rows: int, max_tables: int, max_cols: int) -> list[dict]:
    """Profile a Lakehouse or Warehouse using the Fabric REST API and OneLake direct paths."""
    tables_info = []
    
    try:
        client = fabric.FabricRestClient()
        if item_type == "Lakehouse":
            endpoint = f"/v1/workspaces/{workspace_id}/lakehouses/{item_id}/tables"
        else:
            endpoint = f"/v1/workspaces/{workspace_id}/warehouses/{item_id}/tables"
        
        response = client.get(endpoint)
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code}: {response.text[:200]}")
        
        tables_data = response.json().get("data", [])
        table_names = [t["name"] for t in tables_data]
        print(f"    Listed {len(table_names)} tables via REST API.")
    except Exception as e:
        print(f"    [WARN] Cannot list tables for '{item_name}': {e}")
        return tables_info
    
    if len(table_names) > max_tables:
        print(f"    [SKIP] '{item_name}' has {len(table_names)} tables (>{max_tables}), skipping detail profiling.")
        return [{"name": tn, "columns": [], "sample_values": {}} for tn in table_names]
    
    for table_name in table_names:
        table_info = {"name": table_name, "columns": [], "sample_values": {}}
        try:
            onelake_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{item_id}/Tables/{table_name}"
            sample_df = spark.read.format("delta").load(onelake_path).limit(sample_rows)
            for field in sample_df.schema.fields[:max_cols]:
                table_info["columns"].append(f"{field.name} ({field.dataType.simpleString()})")
            pdf = sample_df.toPandas()
            for col in pdf.columns[:max_cols]:
                vals = pdf[col].dropna().head(sample_rows).tolist()
                table_info["sample_values"][col] = [str(v) for v in vals]
        except Exception as e:
            print(f"    [WARN] Cannot profile table '{item_name}.{table_name}': {e}")
        tables_info.append(table_info)
    return tables_info


def profile_semantic_model(item_name: str, workspace_id: str, sample_rows: int, max_cols: int) -> list[dict]:
    """Profile a Power BI Semantic Model using sempy."""
    tables_info = []
    try:
        tables_df = fabric.list_tables(dataset=item_name, workspace=workspace_id)
    except Exception as e:
        print(f"    [WARN] Cannot list tables for semantic model '{item_name}': {e}")
        return tables_info
    for _, table_row in tables_df.iterrows():
        table_name = table_row["Name"]
        table_info = {"name": table_name, "columns": [], "sample_values": {}}
        try:
            cols_df = fabric.list_columns(dataset=item_name, workspace=workspace_id, table=table_name)
            for _, col_row in cols_df.iterrows():
                table_info["columns"].append(f"{col_row['Column Name']} ({col_row.get('Data Type', 'unknown')})")
            safe_columns = [c.split(" (")[0] for c in table_info["columns"][:max_cols]]
            col_refs = ", ".join([f"'{table_name}'[{c}]" for c in safe_columns])
            dax = f"EVALUATE TOPN({sample_rows}, SELECTCOLUMNS('{table_name}', {col_refs}))"
            result_df = fabric.evaluate_dax(dataset=item_name, dax_string=dax, workspace=workspace_id)
            for col in result_df.columns[:max_cols]:
                vals = result_df[col].dropna().head(sample_rows).tolist()
                table_info["sample_values"][col] = [str(v) for v in vals]
        except Exception as e:
            print(f"    [WARN] Cannot profile semantic model table '{item_name}.{table_name}': {e}")
        tables_info.append(table_info)
    return tables_info


# ---- Kusto / Eventhouse helpers ----

def _get_kql_query_uri(item_id: str, workspace_id: str) -> tuple[str, str]:
    """Get the Kusto query URI and database name for a KQL Database via Fabric REST API."""
    client = fabric.FabricRestClient()
    response = client.get(f"/v1/workspaces/{workspace_id}/kqlDatabases/{item_id}")
    if response.status_code != 200:
        raise Exception(f"HTTP {response.status_code}: {response.text[:200]}")
    db_info = response.json()
    query_uri = db_info.get("properties", {}).get("queryServiceUri") or db_info.get("properties", {}).get("queryUri", "")
    db_name = db_info.get("properties", {}).get("databaseName", db_info.get("displayName", ""))
    if not query_uri:
        raise Exception(f"No queryServiceUri found for KQL Database '{item_id}'")
    return query_uri, db_name


def _get_kusto_token(query_uri: str) -> str:
    """Get a bearer token for the Kusto endpoint."""
    try:
        return notebookutils.credentials.getToken(query_uri)
    except Exception:
        pass
    try:
        return mssparkutils.credentials.getToken(query_uri)
    except Exception:
        pass
    try:
        return notebookutils.credentials.getToken("https://kusto.kusto.windows.net")
    except Exception:
        pass
    return mssparkutils.credentials.getToken("https://kusto.kusto.windows.net")


def _run_kql_mgmt(query_uri: str, db_name: str, command: str) -> dict:
    """Execute a Kusto management command (.show, .create, etc.) via /v1/rest/mgmt."""
    import requests as req
    token = _get_kusto_token(query_uri)
    url = f"{query_uri.rstrip('/')}/v1/rest/mgmt"
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json; charset=utf-8",
        "Accept": "application/json",
        "x-ms-client-request-id": f"fabric-agent-factory;mgmt;{int(time.time())}",
    }
    payload = {"db": db_name, "csl": command}
    resp = req.post(url, headers=headers, json=payload, timeout=60)
    if resp.status_code != 200:
        raise Exception(f"KQL mgmt failed HTTP {resp.status_code}: {resp.text[:300]}")
    return resp.json()


def _run_kql_query(query_uri: str, db_name: str, kql_query: str) -> dict:
    """Execute a KQL data query via /v1/rest/query."""
    import requests as req
    token = _get_kusto_token(query_uri)
    url = f"{query_uri.rstrip('/')}/v1/rest/query"
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json; charset=utf-8",
        "Accept": "application/json",
        "x-ms-client-request-id": f"fabric-agent-factory;query;{int(time.time())}",
    }
    payload = {"db": db_name, "csl": kql_query}
    resp = req.post(url, headers=headers, json=payload, timeout=60)
    if resp.status_code != 200:
        raise Exception(f"KQL query failed HTTP {resp.status_code}: {resp.text[:300]}")
    return resp.json()


def _parse_kql_response(kql_result: dict) -> pd.DataFrame:
    """Parse a Kusto REST API response into a pandas DataFrame."""
    frames = kql_result.get("Tables", kql_result.get("tables", []))
    if not frames:
        return pd.DataFrame()
    table = frames[0]
    columns = [col["ColumnName"] for col in table.get("Columns", table.get("columns", []))]
    rows = table.get("Rows", table.get("rows", []))
    return pd.DataFrame(rows, columns=columns)


def profile_kql_database(item_name: str, item_id: str, workspace_id: str, sample_rows: int, max_cols: int) -> list[dict]:
    """Profile a KQL Database (Eventhouse) using Fabric REST API + Kusto REST endpoints."""
    tables_info = []
    
    # Step 1: Get the Kusto query URI
    try:
        query_uri, db_name = _get_kql_query_uri(item_id, workspace_id)
        print(f"    KQL endpoint: {query_uri[:60]}... db: {db_name}")
    except Exception as e:
        print(f"    [WARN] Cannot get KQL query URI for '{item_name}': {e}")
        return tables_info
    
    # Step 2: List tables via management command → /v1/rest/mgmt
    try:
        result = _run_kql_mgmt(query_uri, db_name, ".show tables | project TableName")
        tables_df = _parse_kql_response(result)
        if tables_df.empty:
            print(f"    No tables found in KQL Database '{item_name}'.")
            return tables_info
        table_names = tables_df.iloc[:, 0].tolist()
        print(f"    Listed {len(table_names)} tables via Kusto mgmt API.")
    except Exception as e:
        print(f"    [WARN] Cannot list KQL tables for '{item_name}': {e}")
        return tables_info
    
    # Step 3: Profile each table — schema via mgmt, sample via query
    for table_name in table_names:
        table_info = {"name": table_name, "columns": [], "sample_values": {}}
        
        # Get schema via management command
        try:
            schema_result = _run_kql_mgmt(
                query_uri, db_name,
                f".show table {table_name} cslschema"
            )
            schema_df = _parse_kql_response(schema_result)
            if not schema_df.empty and "Schema" in schema_df.columns:
                # cslschema returns Schema as "col1:type1,col2:type2,..."
                schema_str = str(schema_df["Schema"].iloc[0])
                for col_def in schema_str.split(","):
                    if ":" in col_def:
                        col_name, col_type = col_def.split(":", 1)
                        table_info["columns"].append(f"{col_name.strip()} ({col_type.strip()})")
            elif not schema_df.empty:
                # Fallback: try to use whatever columns are returned
                for _, row in schema_df.iterrows():
                    col_name = str(row.iloc[0]) if len(row) > 0 else "unknown"
                    col_type = str(row.iloc[1]) if len(row) > 1 else "unknown"
                    table_info["columns"].append(f"{col_name} ({col_type})")
        except Exception as e:
            print(f"    [WARN] Cannot get schema for KQL table '{table_name}': {e}")
        
        # Sample rows via data query → /v1/rest/query
        try:
            sample_result = _run_kql_query(query_uri, db_name, f"{table_name} | take {sample_rows}")
            sample_df = _parse_kql_response(sample_result)
            for col in sample_df.columns[:max_cols]:
                vals = sample_df[col].dropna().head(sample_rows).tolist()
                table_info["sample_values"][col] = [str(v) for v in vals]
        except Exception as e:
            print(f"    [WARN] Cannot sample KQL table '{table_name}': {e}")
        
        tables_info.append(table_info)
    
    return tables_info

In [ ]:
# ==============================================================
# ALL profiling helpers — run this cell to load latest versions
# ==============================================================
import uuid as _uuid

def profile_lakehouse_or_warehouse(item_name, item_id, item_type, workspace_id, sample_rows, max_tables, max_cols):
    tables_info = []
    try:
        client = fabric.FabricRestClient()
        endpoint = f"/v1/workspaces/{workspace_id}/{'lakehouses' if item_type == 'Lakehouse' else 'warehouses'}/{item_id}/tables"
        response = client.get(endpoint)
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code}: {response.text[:200]}")
        tables_data = response.json().get("data", [])
        table_names = [t["name"] for t in tables_data]
        print(f"    Listed {len(table_names)} tables via REST API.")
    except Exception as e:
        print(f"    [WARN] Cannot list tables for '{item_name}': {e}")
        return tables_info
    if len(table_names) > max_tables:
        print(f"    [SKIP] '{item_name}' has {len(table_names)} tables (>{max_tables}), skipping.")
        return [{"name": tn, "columns": [], "sample_values": {}} for tn in table_names]
    for table_name in table_names:
        table_info = {"name": table_name, "columns": [], "sample_values": {}}
        try:
            onelake_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{item_id}/Tables/{table_name}"
            sample_df = spark.read.format("delta").load(onelake_path).limit(sample_rows)
            for field in sample_df.schema.fields[:max_cols]:
                table_info["columns"].append(f"{field.name} ({field.dataType.simpleString()})")
            pdf = sample_df.toPandas()
            for col in pdf.columns[:max_cols]:
                vals = pdf[col].dropna().head(sample_rows).tolist()
                table_info["sample_values"][col] = [str(v) for v in vals]
        except Exception as e:
            print(f"    [WARN] Cannot profile table '{item_name}.{table_name}': {e}")
        tables_info.append(table_info)
    return tables_info


def profile_semantic_model(item_name, workspace_id, sample_rows, max_cols):
    tables_info = []
    try:
        tables_df = fabric.list_tables(dataset=item_name, workspace=workspace_id)
    except Exception as e:
        print(f"    [WARN] Cannot list tables for semantic model '{item_name}': {e}")
        return tables_info
    for _, table_row in tables_df.iterrows():
        table_name = table_row["Name"]
        table_info = {"name": table_name, "columns": [], "sample_values": {}}
        try:
            cols_df = fabric.list_columns(dataset=item_name, workspace=workspace_id, table=table_name)
            for _, col_row in cols_df.iterrows():
                table_info["columns"].append(f"{col_row['Column Name']} ({col_row.get('Data Type', 'unknown')})")
            safe_columns = [c.split(" (")[0] for c in table_info["columns"][:max_cols]]
            col_refs = ", ".join([f"'{table_name}'[{c}]" for c in safe_columns])
            dax = f"EVALUATE TOPN({sample_rows}, SELECTCOLUMNS('{table_name}', {col_refs}))"
            result_df = fabric.evaluate_dax(dataset=item_name, dax_string=dax, workspace=workspace_id)
            for col in result_df.columns[:max_cols]:
                vals = result_df[col].dropna().head(sample_rows).tolist()
                table_info["sample_values"][col] = [str(v) for v in vals]
        except Exception as e:
            print(f"    [WARN] Cannot profile semantic model table '{item_name}.{table_name}': {e}")
        tables_info.append(table_info)
    return tables_info


# ---- Kusto / Eventhouse helpers ----

def _get_kql_query_uri(item_id, workspace_id):
    client = fabric.FabricRestClient()
    response = client.get(f"/v1/workspaces/{workspace_id}/kqlDatabases/{item_id}")
    if response.status_code != 200:
        raise Exception(f"HTTP {response.status_code}: {response.text[:200]}")
    db_info = response.json()
    query_uri = db_info.get("properties", {}).get("queryServiceUri") or db_info.get("properties", {}).get("queryUri", "")
    db_name = db_info.get("properties", {}).get("databaseName", db_info.get("displayName", ""))
    if not query_uri:
        raise Exception(f"No queryServiceUri found for KQL DB '{item_id}'")
    return query_uri, db_name


def _get_kusto_token(query_uri):
    for fn_name, fn in [
        ("notebookutils", lambda u: notebookutils.credentials.getToken(u)),
        ("mssparkutils", lambda u: mssparkutils.credentials.getToken(u)),
    ]:
        for audience in [query_uri, "https://kusto.fabric.microsoft.com", "https://kusto.kusto.windows.net"]:
            try:
                return fn(audience)
            except Exception:
                pass
    raise RuntimeError(f"Cannot acquire Kusto token for {query_uri}")


def _kusto_post(query_uri, db_name, csl, endpoint_path):
    """Send a POST to the Kusto REST API (mgmt or query endpoint)."""
    import requests as _req
    token = _get_kusto_token(query_uri)
    url = f"{query_uri.rstrip('/')}{endpoint_path}"
    body = json.dumps({"db": db_name, "csl": csl}).encode("utf-8")
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json; charset=utf-8",
        "Accept": "application/json",
        "x-ms-client-request-id": f"fabric-agent;{str(_uuid.uuid4())}",
    }
    resp = _req.post(url, data=body, headers=headers, timeout=60)
    if resp.status_code != 200:
        raise Exception(f"Kusto {endpoint_path} HTTP {resp.status_code}: {resp.text[:500]}")
    return resp.json()


def _parse_kql(result):
    frames = result.get("Tables", result.get("tables", []))
    if not frames:
        return pd.DataFrame()
    t = frames[0]
    cols = [c["ColumnName"] for c in t.get("Columns", t.get("columns", []))]
    rows = t.get("Rows", t.get("rows", []))
    return pd.DataFrame(rows, columns=cols)


def profile_kql_database(item_name, item_id, workspace_id, sample_rows, max_cols):
    """Profile a KQL Database (Eventhouse) via Kusto REST API."""
    tables_info = []

    # Step 1: Get query URI
    try:
        query_uri, db_name = _get_kql_query_uri(item_id, workspace_id)
        print(f"    KQL endpoint: {query_uri[:60]}... db: {db_name}")
    except Exception as e:
        print(f"    [WARN] Cannot get KQL info for '{item_name}': {e}")
        return tables_info

    # Step 2: List tables via management endpoint
    try:
        r = _kusto_post(query_uri, db_name, ".show tables | project TableName", "/v1/rest/mgmt")
        tdf = _parse_kql(r)
        if tdf.empty:
            print(f"    No tables found.")
            return tables_info
        table_names = tdf.iloc[:, 0].tolist()
        print(f"    Listed {len(table_names)} tables via Kusto mgmt API.")
    except Exception as e:
        print(f"    [WARN] Cannot list KQL tables for '{item_name}': {e}")
        return tables_info

    # Step 3: Profile each table
    for table_name in table_names:
        table_info = {"name": table_name, "columns": [], "sample_values": {}}

        # 3a: Schema via management command
        try:
            sr = _kusto_post(query_uri, db_name, f".show table {table_name} cslschema", "/v1/rest/mgmt")
            sdf = _parse_kql(sr)
            if not sdf.empty and "Schema" in sdf.columns:
                for col_def in str(sdf["Schema"].iloc[0]).split(","):
                    if ":" in col_def:
                        cn, ct = col_def.split(":", 1)
                        table_info["columns"].append(f"{cn.strip()} ({ct.strip()})")
            print(f"    ✓ Schema for '{table_name}': {len(table_info['columns'])} columns")
        except Exception as e:
            print(f"    [WARN] Schema failed for '{table_name}': {e}")

        # 3b: Sample rows via query endpoint
        try:
            qr = _kusto_post(query_uri, db_name, f"{table_name} | take {sample_rows}", "/v1/rest/query")
            qdf = _parse_kql(qr)
            for col in qdf.columns[:max_cols]:
                vals = qdf[col].dropna().head(sample_rows).tolist()
                table_info["sample_values"][col] = [str(v) for v in vals]
            # If we got sample data but no schema yet, infer columns from sample
            if not table_info["columns"] and not qdf.empty:
                table_info["columns"] = [f"{c} (unknown)" for c in qdf.columns[:max_cols]]
            print(f"    ✓ Sampled '{table_name}': {len(qdf)} rows")
        except Exception as e:
            print(f"    [WARN] Sample failed for '{table_name}': {e}")

        tables_info.append(table_info)

    return tables_info


# ----- Profile all items -----

def profile_all_items(inventory_df):
    profiled_items = []
    for idx, row in inventory_df.iterrows():
        item_name = row["item_name"]
        item_id = row["item_id"]
        item_type = row["item_type"]
        ws_id = row["workspace_id"]
        ws_name = row["workspace_name"]
        print(f"\n[{idx+1}/{len(inventory_df)}] Profiling {item_type} '{item_name}' in '{ws_name}'...")
        if item_type in ("Lakehouse", "Warehouse"):
            tables = profile_lakehouse_or_warehouse(item_name, item_id, item_type, ws_id, SAMPLE_ROWS, MAX_TABLES_PER_ITEM, MAX_COLUMNS_IN_SAMPLE)
        elif item_type == "SemanticModel":
            tables = profile_semantic_model(item_name, ws_id, SAMPLE_ROWS, MAX_COLUMNS_IN_SAMPLE)
        elif item_type == "KQLDatabase":
            tables = profile_kql_database(item_name, item_id, ws_id, SAMPLE_ROWS, MAX_COLUMNS_IN_SAMPLE)
        else:
            print(f"    [SKIP] Unsupported type '{item_type}'")
            continue
        profiled_items.append({
            "item_id": item_id, "item_name": item_name, "item_type": item_type,
            "workspace_id": ws_id, "workspace_name": ws_name,
            "tables": tables, "table_count": len(tables),
        })
        print(f"    Found {len(tables)} tables.")
    print(f"\nProfiling complete: {len(profiled_items)} items profiled.")
    return profiled_items

# Run profiling
profiled_items = profile_all_items(item_inventory_df)

# Summary
summary = pd.DataFrame([{
    "item_name": p["item_name"], "item_type": p["item_type"],
    "workspace": p["workspace_name"], "tables": p["table_count"],
} for p in profiled_items])
display(summary)

---
## Phase 2.5 — Table-Level Context Extraction & Similarity

For each table individually, the LLM extracts a semantic description and domain tags.
Then a programmatic column-name similarity check detects cross-item relationships
(e.g., an Eventhouse table with the same columns as a Lakehouse table).

This enables Phase 3 to cluster by **table content**, not by Fabric item.

In [ ]:
# ==============================================================
# Phase 2.5 — Per-Table Context Extraction & Similarity Matrix
# ==============================================================
import requests as req
import re as _re

# Ensure LLM access is available for context extraction
def _get_aoai_token():
    try:
        return aoai_token
    except NameError:
        pass
    token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    resp = req.post(token_url, data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": "https://cognitiveservices.azure.com/.default"
    }, timeout=30)
    resp.raise_for_status()
    return resp.json()["access_token"]

def call_azure_openai(prompt):
    token = _get_aoai_token()
    url = f"https://{resource_name}.openai.azure.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    payload = {
        "model": deployment_name,
        "messages": [
            {"role": "system", "content": "You are a data architecture expert. Respond only with valid JSON."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.3,
        "max_tokens": 16000,
        "response_format": {"type": "json_object"},
    }
    resp = req.post(url, headers=headers, json=payload, timeout=180)
    if resp.status_code != 200:
        raise Exception(f"Azure OpenAI HTTP {resp.status_code}: {resp.text[:500]}")
    return json.loads(resp.json()["choices"][0]["message"]["content"])


def _flatten_tables(profiled_items):
    flat = []
    for p in profiled_items:
        for t in p["tables"]:
            flat.append({
                "table_name": t["name"],
                "item_name": p["item_name"],
                "item_id": p["item_id"],
                "item_type": p["item_type"],
                "workspace_id": p["workspace_id"],
                "workspace_name": p["workspace_name"],
                "columns": t.get("columns", []),
                "sample_values": t.get("sample_values", {}),
            })
    return flat


def _build_context_prompt(tables_batch):
    tables_text = ""
    for idx, t in enumerate(tables_batch):
        tables_text += f"\n### Table {idx+1}: {t['table_name']} (from {t['item_name']}, type: {t['item_type']})\n"
        if t["columns"]:
            tables_text += f"  Columns: {', '.join(t['columns'][:15])}\n"
        if t["sample_values"]:
            for col, vals in list(t["sample_values"].items())[:5]:
                tables_text += f"  Sample {col}: {vals[:3]}\n"
    
    return f"""Analyze each of the {len(tables_batch)} tables below. For EACH table provide:
1. A 1-2 sentence description of what data this table contains
2. Domain tags (3-7 tags) capturing the business domain
3. Whether it is a fact, dimension, event/streaming, or reference table

IMPORTANT: Return EXACTLY {len(tables_batch)} entries in the same order as listed above.
Copy table_name and item_name EXACTLY as shown — do not modify them.

{tables_text}

Respond with valid JSON containing exactly {len(tables_batch)} entries:
{{
  "tables": [
    {{
      "table_name": "EXACT table name as shown above",
      "item_name": "EXACT item name as shown above",
      "description": "Short description of the table content",
      "domain_tags": ["tag1", "tag2", "tag3"],
      "table_role": "fact|dimension|event|reference"
    }}
  ]
}}"""


def _match_context(t, contexts, idx):
    for c in contexts:
        if c.get("table_name") == t["table_name"] and c.get("item_name") == t["item_name"]:
            return c
    for c in contexts:
        if (c.get("table_name", "").lower() == t["table_name"].lower() and 
            c.get("item_name", "").lower() == t["item_name"].lower()):
            return c
    for c in contexts:
        if c.get("table_name", "").lower() == t["table_name"].lower():
            return c
    if idx < len(contexts):
        return contexts[idx]
    return None


def extract_table_contexts(flat_tables, call_llm_fn, batch_size=10):
    all_contexts = []
    for i in range(0, len(flat_tables), batch_size):
        batch = flat_tables[i:i + batch_size]
        print(f"  Extracting context for tables {i+1}-{min(i+batch_size, len(flat_tables))} of {len(flat_tables)}...")
        prompt = _build_context_prompt(batch)
        try:
            result = call_llm_fn(prompt)
            contexts = result.get("tables", [])
            if len(contexts) != len(batch):
                print(f"    [WARN] LLM returned {len(contexts)} contexts for {len(batch)} tables")
            for idx, t in enumerate(batch):
                matched = _match_context(t, contexts, idx)
                if matched:
                    t["description"] = matched.get("description", "")
                    t["domain_tags"] = matched.get("domain_tags", [])
                    t["table_role"] = matched.get("table_role", "unknown")
                else:
                    t["description"] = ""
                    t["domain_tags"] = []
                    t["table_role"] = "unknown"
                all_contexts.append(t)
                tags_str = ', '.join(t['domain_tags'][:5]) if t['domain_tags'] else 'no tags'
                print(f"    ✓ {t['item_name']}.{t['table_name']}: [{tags_str}] ({t['table_role']})")
        except Exception as e:
            print(f"    [WARN] Context extraction failed for batch: {e}")
            for t in batch:
                t["description"] = ""
                t["domain_tags"] = []
                t["table_role"] = "unknown"
                all_contexts.append(t)
    return all_contexts


def _normalize_col_name(name):
    """Normalize a column name for fuzzy matching.
    Strips common prefixes (lpep_, tpep_, etc.), lowercases, removes underscores."""
    n = name.split(" (")[0].lower().strip()
    # Remove common prefixes from taxi/transport datasets
    n = _re.sub(r'^(lpep_|tpep_|green_|yellow_|pickup_|dropoff_|pu|do)', '', n)
    # Also normalize _id, id suffixes
    n = _re.sub(r'_?id$', '_id', n)
    return n


def compute_column_similarity(flat_tables, min_similarity=0.15, min_overlap=2):
    """Compute column similarity using BOTH exact and normalized (fuzzy) column name matching."""
    table_cols_raw = {}
    table_cols_norm = {}
    for t in flat_tables:
        key = f"{t['item_name']}.{t['table_name']}"
        raw = {c.split(" (")[0].lower().strip() for c in t.get("columns", [])}
        norm = {_normalize_col_name(c) for c in t.get("columns", [])}
        table_cols_raw[key] = raw
        table_cols_norm[key] = norm
    
    similarities = []
    keys = list(table_cols_raw.keys())
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            raw_a, raw_b = table_cols_raw[keys[i]], table_cols_raw[keys[j]]
            norm_a, norm_b = table_cols_norm[keys[i]], table_cols_norm[keys[j]]
            
            if not raw_a or not raw_b:
                continue
            
            # Exact match
            exact_overlap = raw_a & raw_b
            exact_union = raw_a | raw_b
            exact_sim = len(exact_overlap) / len(exact_union) if exact_union else 0
            
            # Fuzzy/normalized match
            fuzzy_overlap = norm_a & norm_b
            fuzzy_union = norm_a | norm_b
            fuzzy_sim = len(fuzzy_overlap) / len(fuzzy_union) if fuzzy_union else 0
            
            # Use the better of the two
            best_sim = max(exact_sim, fuzzy_sim)
            best_overlap = exact_overlap if exact_sim >= fuzzy_sim else fuzzy_overlap
            match_type = "exact" if exact_sim >= fuzzy_sim else "fuzzy"
            
            if best_sim >= min_similarity and len(best_overlap) >= min_overlap:
                similarities.append({
                    "table_a": keys[i],
                    "table_b": keys[j],
                    "overlap_columns": sorted(best_overlap)[:8],
                    "similarity": round(best_sim, 2),
                    "match_type": match_type,
                })
    
    similarities.sort(key=lambda x: x["similarity"], reverse=True)
    return similarities


def compute_tag_similarity(flat_tables, min_overlap=2):
    """Compute domain-tag overlap between tables from DIFFERENT items."""
    tag_hints = []
    for i in range(len(flat_tables)):
        for j in range(i + 1, len(flat_tables)):
            a, b = flat_tables[i], flat_tables[j]
            # Only cross-item comparisons
            if a["item_name"] == b["item_name"]:
                continue
            tags_a = set(t.lower() for t in a.get("domain_tags", []))
            tags_b = set(t.lower() for t in b.get("domain_tags", []))
            if not tags_a or not tags_b:
                continue
            overlap = tags_a & tags_b
            if len(overlap) >= min_overlap:
                tag_hints.append({
                    "table_a": f"{a['item_name']}.{a['table_name']}",
                    "table_b": f"{b['item_name']}.{b['table_name']}",
                    "shared_tags": sorted(overlap),
                    "tag_overlap": len(overlap),
                })
    tag_hints.sort(key=lambda x: x["tag_overlap"], reverse=True)
    return tag_hints


# --- Run Phase 2.5 ---
print("=" * 60)
print("Phase 2.5 — Table Context Extraction & Similarity")
print("=" * 60)

# Step 1: Flatten
flat_tables = _flatten_tables(profiled_items)
print(f"\nFlattened {len(flat_tables)} tables across all items.\n")

# Step 2: LLM context extraction
print("Step 1: Extracting per-table context via LLM...")
table_contexts = extract_table_contexts(flat_tables, call_azure_openai)

# Step 3: Column similarity (exact + fuzzy)
print("\nStep 2: Computing column similarity (exact + fuzzy normalized)...")
similarity_hints = compute_column_similarity(table_contexts)

# Step 4: Domain tag similarity (cross-item only)
print("Step 3: Computing domain-tag similarity (cross-item)...")
tag_hints = compute_tag_similarity(table_contexts)

# Display column similarities
if similarity_hints:
    print(f"\nColumn similarities ({len(similarity_hints)} pairs):")
    for s in similarity_hints[:15]:
        print(f"  {s['table_a']} <-> {s['table_b']}: {s['similarity']:.0%} ({s['match_type']}) [{', '.join(s['overlap_columns'][:5])}]")

# Display tag similarities (cross-item only)
if tag_hints:
    print(f"\nCross-item domain tag matches ({len(tag_hints)} pairs):")
    for t in tag_hints[:10]:
        print(f"  {t['table_a']} <-> {t['table_b']}: {t['tag_overlap']} shared tags [{', '.join(t['shared_tags'][:5])}]")

# Merge both similarity types into one list for the clustering prompt
all_hints = similarity_hints.copy()
# Add tag hints that aren't already covered by column similarity
col_pairs = {(s["table_a"], s["table_b"]) for s in similarity_hints}
for th in tag_hints:
    pair = (th["table_a"], th["table_b"])
    rpair = (th["table_b"], th["table_a"])
    if pair not in col_pairs and rpair not in col_pairs:
        all_hints.append({
            "table_a": th["table_a"],
            "table_b": th["table_b"],
            "overlap_columns": th["shared_tags"],
            "similarity": th["tag_overlap"] / 10,  # Normalize to 0-1 range
            "match_type": "domain_tags",
        })
# Re-assign for use in Phase 3
similarity_hints = all_hints
print(f"\nTotal similarity hints for clustering: {len(similarity_hints)}")

# Display summary table
ctx_df = pd.DataFrame([{
    "item": t["item_name"], "table": t["table_name"], "role": t.get("table_role", "?"),
    "tags": ", ".join(t.get("domain_tags", [])[:5]),
    "desc": (t.get("description", "")[:80] + "...") if len(t.get("description", "")) > 80 else t.get("description", ""),
} for t in table_contexts])
print("\nTable Contexts:")
display(ctx_df)

tagged = sum(1 for t in table_contexts if t.get("domain_tags"))
print(f"\n{tagged}/{len(table_contexts)} tables have domain tags.")

---
## Phase 3 — LLM-based Topic Clustering

Sends the profiled schema + sample data to Azure OpenAI to intelligently cluster data items into thematic Data Agents.

In [ ]:
# ============================================================
# LLM Connection via Service Principal (Client Credentials)
# ============================================================
import json
import requests as req


# ⚠️ SECURITY WARNING: Hardcoding client_secret is NOT recommended.
# Consider loading it from Key Vault instead:
#   client_secret = mssparkutils.credentials.getSecret("<vault-url>", "<secret-name>")

# Get token via OAuth2 client credentials flow
token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
token_resp = req.post(token_url, data={
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": "https://cognitiveservices.azure.com/.default"
}, timeout=30)
token_resp.raise_for_status()
aoai_token = token_resp.json()["access_token"]
print("✓ Token acquired via Service Principal (client credentials flow)")

# Test the connection
url = f"https://{resource_name}.openai.azure.com/openai/v1/chat/completions"

payload = {
    "model": deployment_name,
    "messages": [
        {"role": "user", "content": "Reply with exactly: connection successful"}
    ],
    "max_tokens": 20
}

headers = {
    "Authorization": f"Bearer {aoai_token}",
    "Content-Type": "application/json"
}

resp = req.post(url, headers=headers, json=payload, timeout=30)

print("HTTP", resp.status_code)
print(resp.text)

if resp.status_code == 200:
    print("\n✅ Connection successful! LLM is reachable.")
else:
    print("\n❌ Connection failed. Check credentials and endpoint.")

In [ ]:
def build_clustering_prompt(table_contexts, similarity_hints, max_ds_per_agent, max_examples):
    """Build the LLM prompt using per-table contexts and similarity hints."""
    
    tables_text = ""
    for t in table_contexts:
        tables_text += f"\n### Table: {t['table_name']}"
        tables_text += f"\n  Source: {t['item_name']} ({t['item_type']}) in workspace '{t['workspace_name']}'"
        tables_text += f"\n  Role: {t.get('table_role', 'unknown')}"
        tables_text += f"\n  Description: {t.get('description', 'N/A')}"
        tables_text += f"\n  Tags: {', '.join(t.get('domain_tags', []))}"
        if t["columns"]:
            tables_text += f"\n  Columns: {', '.join(t['columns'][:15])}"
        if t["sample_values"]:
            for col, vals in list(t["sample_values"].items())[:5]:
                tables_text += f"\n  Sample {col}: {vals[:3]}"
        tables_text += "\n"
    
    sim_text = ""
    if similarity_hints:
        sim_text = "\n\nCROSS-TABLE RELATIONSHIPS DETECTED:\n"
        for s in similarity_hints[:20]:
            mtype = s.get("match_type", "exact")
            sim_text += f"  - {s['table_a']} <-> {s['table_b']}: {s['similarity']:.0%} overlap ({mtype}) [{', '.join(s['overlap_columns'][:5])}]\n"
        sim_text += "\nThese relationships mean these tables belong to the SAME domain → group into the SAME agent!\n"
        sim_text += "Pay special attention to CROSS-ITEM matches (different item_name) — these are the most valuable signals.\n"
    
    prompt = f"""You are an expert data architect. Analyze the tables below and group them into thematic Data Agents.

CRITICAL RULES — READ CAREFULLY:

1. GROUP BY CONTENT, NOT BY ITEM.
   - Tables from different Fabric items that cover the same domain MUST be in the SAME agent.
   - Tables from the SAME item that cover different domains MUST be in DIFFERENT agents.
   - SPECIFICALLY: If a KQLDatabase/Eventhouse has a table with taxi/trip data AND a Lakehouse also has taxi/trip data → they MUST be in the SAME agent together. The Eventhouse provides real-time data, the Lakehouse provides historical data — combining them gives the agent both perspectives.

2. USE THE CROSS-TABLE RELATIONSHIPS below. Tables with overlapping columns or shared domain tags almost certainly belong together — ESPECIALLY when they are from different items (cross-item matches).

3. A table CAN appear in MULTIPLE agents if relevant to multiple domains.
   - Example: Weather data in both a transportation agent AND a supply chain agent.
   - Example: Public holidays in both retail analytics AND transportation analytics.

4. NO DUPLICATE ITEMS per agent. Each item_name MUST appear AT MOST ONCE in an agent's data_sources array. Put ALL tables from the same item into ONE data_source entry with all selected_tables listed together.
   - WRONG: two entries with item_name="New_York_Taxi_LH" in the same agent
   - RIGHT: one entry with item_name="New_York_Taxi_LH" and selected_tables listing all relevant tables

5. Each agent can have at most {max_ds_per_agent} data sources.

6. For each agent provide:
   - Name, Description (2-3 sentences)
   - Data sources with selected_tables
   - System instructions referencing actual table/column names
   - Example queries: up to {max_examples} per source (SQL for Lakehouse/Warehouse, KQL for KQLDatabase, NONE for SemanticModel)

7. Every table MUST be assigned to at least one agent.

TABLES TO ANALYZE:
{tables_text}
{sim_text}

Respond ONLY with valid JSON:
{{
  "agents": [
    {{
      "name": "Agent Name",
      "description": "What this agent covers.",
      "data_sources": [
        {{
          "item_name": "exact item name",
          "item_type": "Lakehouse|Warehouse|SemanticModel|KQLDatabase",
          "workspace_name": "exact workspace name",
          "selected_tables": ["table1", "table2"],
          "data_instructions": "Per-datasource instructions.",
          "example_queries": [
            {{"question": "...", "sql": "..."}}
          ]
        }}
      ],
      "system_instructions": "Detailed system instructions..."
    }}
  ]
}}
"""
    return prompt


def merge_duplicate_datasources(plan):
    """Post-processing: merge duplicate item_name entries within the same agent."""
    for agent in plan.get("agents", []):
        sources = agent.get("data_sources", [])
        merged = {}
        for ds in sources:
            key = ds.get("item_name", "")
            if key in merged:
                # Merge selected_tables
                existing_tables = set(merged[key].get("selected_tables", []))
                new_tables = set(ds.get("selected_tables", []))
                merged[key]["selected_tables"] = sorted(existing_tables | new_tables)
                # Merge instructions
                existing_instr = merged[key].get("data_instructions", "")
                new_instr = ds.get("data_instructions", "")
                if new_instr and new_instr not in existing_instr:
                    merged[key]["data_instructions"] = f"{existing_instr} {new_instr}".strip()
                # Merge example queries
                existing_eq = merged[key].get("example_queries", [])
                new_eq = ds.get("example_queries", [])
                seen_q = {eq.get("question", "") for eq in existing_eq}
                for eq in new_eq:
                    if eq.get("question", "") not in seen_q:
                        existing_eq.append(eq)
                merged[key]["example_queries"] = existing_eq
            else:
                merged[key] = ds.copy()
        agent["data_sources"] = list(merged.values())
    return plan


def get_aoai_token():
    try:
        return aoai_token
    except NameError:
        pass
    token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    token_resp = req.post(token_url, data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": "https://cognitiveservices.azure.com/.default"
    }, timeout=30)
    token_resp.raise_for_status()
    return token_resp.json()["access_token"]


def call_azure_openai(prompt):
    import requests as req
    token = get_aoai_token()
    url = f"https://{resource_name}.openai.azure.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    payload = {
        "model": deployment_name,
        "messages": [
            {"role": "system", "content": "You are a data architecture expert. Respond only with valid JSON."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.3,
        "max_tokens": 16000,
        "response_format": {"type": "json_object"},
    }
    resp = req.post(url, headers=headers, json=payload, timeout=180)
    if resp.status_code != 200:
        raise Exception(f"Azure OpenAI HTTP {resp.status_code}: {resp.text[:500]}")
    return json.loads(resp.json()["choices"][0]["message"]["content"])


def validate_agent_plan(plan, profiled_items):
    warnings = []
    known_items = {p["item_name"] for p in profiled_items}
    assigned_items = set()
    
    for agent in plan.get("agents", []):
        name = agent.get("name", "<unnamed>")
        sources = agent.get("data_sources", [])
        if len(sources) > MAX_DATASOURCES_PER_AGENT:
            warnings.append(f"Agent '{name}' has {len(sources)} data sources (max {MAX_DATASOURCES_PER_AGENT}). Will be split.")
        for ds in sources:
            ds_name = ds.get("item_name", "")
            assigned_items.add(ds_name)
            if ds_name not in known_items:
                warnings.append(f"Agent '{name}' references unknown item '{ds_name}'.")
            if ds.get("item_type") == "SemanticModel" and ds.get("example_queries"):
                ds["example_queries"] = []
                warnings.append(f"Agent '{name}', source '{ds_name}': Removed example queries (SemanticModel).")
    
    unassigned = known_items - assigned_items
    if unassigned:
        warnings.append(f"Unassigned items: {unassigned}")
    return warnings


def split_oversized_agents(plan):
    new_agents = []
    for agent in plan.get("agents", []):
        sources = agent.get("data_sources", [])
        if len(sources) <= MAX_DATASOURCES_PER_AGENT:
            new_agents.append(agent)
        else:
            for i in range(0, len(sources), MAX_DATASOURCES_PER_AGENT):
                chunk = sources[i:i + MAX_DATASOURCES_PER_AGENT]
                part_num = (i // MAX_DATASOURCES_PER_AGENT) + 1
                new_agents.append({**agent, "name": f"{agent['name']} Part {part_num}", "data_sources": chunk})
    plan["agents"] = new_agents
    return plan

In [ ]:
# Build prompt using table contexts + similarity hints from Phase 2.5
print("Building table-level clustering prompt...")
prompt = build_clustering_prompt(table_contexts, similarity_hints, MAX_DATASOURCES_PER_AGENT, MAX_EXAMPLES_PER_DATASOURCE)
print(f"Prompt length: {len(prompt):,} characters")

print("\nCalling Azure OpenAI for context-aware clustering...")
agent_plan = call_azure_openai(prompt)
print(f"\nLLM proposed {len(agent_plan.get('agents', []))} agents.")

# Post-processing: merge duplicate datasources within agents
print("Merging duplicate datasource entries...")
agent_plan = merge_duplicate_datasources(agent_plan)

# Validate
warnings = validate_agent_plan(agent_plan, profiled_items)
if warnings:
    print("\nValidation warnings:")
    for w in warnings:
        print(f"  ⚠ {w}")

# Split oversized agents
agent_plan = split_oversized_agents(agent_plan)
print(f"\nFinal plan: {len(agent_plan['agents'])} agents after validation.")

---
## Human Review Checkpoint

Review the proposed agents below. After reviewing, set `DRY_RUN = False` in Cell 0 and re-run Phase 4 to create the agents.

In [ ]:
def display_agent_plan(plan: dict):
    """Display the proposed agent plan in a human-readable format."""
    agents = plan.get("agents", [])
    
    for i, agent in enumerate(agents, 1):
        print(f"{'='*80}")
        print(f"AGENT {i}: {agent['name']}")
        print(f"{'='*80}")
        print(f"Description: {agent.get('description', 'N/A')}")
        print(f"\nSystem Instructions (preview):")
        instructions = agent.get('system_instructions', 'N/A')
        print(f"  {instructions[:500]}{'...' if len(instructions) > 500 else ''}")
        print(f"\nData Sources ({len(agent.get('data_sources', []))}):\n")
        
        for ds in agent.get("data_sources", []):
            print(f"  📦 {ds['item_name']} ({ds['item_type']}) — Workspace: {ds.get('workspace_name', 'N/A')}")
            print(f"     Tables: {', '.join(ds.get('selected_tables', []))}")
            di = ds.get('data_instructions', '')
            if di:
                print(f"     Data Instructions: {di[:200]}{'...' if len(di) > 200 else ''}")
            examples = ds.get('example_queries', [])
            if examples:
                print(f"     Example Queries ({len(examples)}):")
                for eq in examples[:3]:  # Show first 3
                    print(f"       Q: {eq.get('question', '')}")
                    print(f"       A: {eq.get('sql', '')[:120]}")
                if len(examples) > 3:
                    print(f"       ... and {len(examples) - 3} more")
            print()
    
    print(f"\n{'='*80}")
    print(f"TOTAL: {len(agents)} agents planned.")
    if DRY_RUN:
        print("\n👉 DRY_RUN is ON. Set DRY_RUN = False in Cell 0 and re-run Phase 4 to create agents.")
    else:
        print("\n🚀 DRY_RUN is OFF. Phase 4 will CREATE and PUBLISH these agents.")


display_agent_plan(agent_plan)

#### Optional: Edit the plan manually

Uncomment and modify the cell below to adjust agent names, merge/split agents, or change instructions before creation.

In [ ]:
# # Example: Rename an agent
# agent_plan["agents"][0]["name"] = "Custom Fraud & Payment Agent"
#
# # Example: Merge agents 0 and 1
# merged = agent_plan["agents"][0]
# merged["data_sources"] += agent_plan["agents"][1]["data_sources"]
# merged["name"] = "Merged Agent"
# agent_plan["agents"] = [merged] + agent_plan["agents"][2:]
#
# # Example: Remove an agent
# agent_plan["agents"] = [a for a in agent_plan["agents"] if a["name"] != "Unwanted Agent"]
#
# # Re-validate after edits
# agent_plan = split_oversized_agents(agent_plan)
# display_agent_plan(agent_plan)

---
## Phase 4 — Data Agent Creation & Configuration

Creates, configures, and publishes Data Agents using the `fabric-data-agent-sdk`.

In [ ]:
# Map item_type strings to SDK datasource type parameter
ITEM_TYPE_TO_SDK_TYPE = {
    "Lakehouse": "lakehouse",
    "Warehouse": "warehouse",
    "SemanticModel": "semanticmodel",
    "KQLDatabase": "kqldatabase",
}


def create_agents_from_plan(plan: dict, dry_run: bool) -> list[dict]:
    """Create and configure Data Agents based on the validated plan."""
    created_agents = []
    agents = plan.get("agents", [])
    
    for i, agent_def in enumerate(agents, 1):
        agent_name = f"{AGENT_NAME_PREFIX} - {agent_def['name']}"
        # Sanitize name (remove special chars that might cause issues)
        agent_name = re.sub(r'[^\w\s\-]', '', agent_name).strip()[:256]
        
        print(f"\n{'='*60}")
        print(f"[{i}/{len(agents)}] Agent: {agent_name}")
        print(f"{'='*60}")
        
        if dry_run:
            print("  [DRY RUN] Would create agent. Skipping.")
            created_agents.append({
                "agent_name": agent_name,
                "status": "dry_run",
                "data_sources": len(agent_def.get("data_sources", [])),
                "description": agent_def.get("description", ""),
            })
            continue
        
        # Step 1: Create the agent
        try:
            print(f"  Creating agent '{agent_name}'...")
            data_agent = create_data_agent(agent_name)
            print(f"  ✓ Agent created.")
        except Exception as e:
            print(f"  ✗ Failed to create agent: {e}")
            created_agents.append({"agent_name": agent_name, "status": f"create_failed: {e}"})
            continue
        
        # Step 2: Set system-level instructions
        try:
            sys_instructions = agent_def.get("system_instructions", "")
            if sys_instructions:
                # Truncate to 15,000 chars (SDK limit)
                data_agent.update_configuration(instructions=sys_instructions[:15000])
                print(f"  ✓ System instructions set ({len(sys_instructions)} chars).")
        except Exception as e:
            print(f"  ⚠ Failed to set system instructions: {e}")
        
        # Step 3: Add data sources and configure each
        for ds_def in agent_def.get("data_sources", []):
            ds_item_name = ds_def["item_name"]
            ds_item_type = ds_def["item_type"]
            sdk_type = ITEM_TYPE_TO_SDK_TYPE.get(ds_item_type, ds_item_type.lower())
            
            try:
                print(f"  Adding datasource '{ds_item_name}' ({sdk_type})...")
                data_agent.add_datasource(ds_item_name, type=sdk_type)
                print(f"  ✓ Datasource added.")
            except Exception as e:
                print(f"  ✗ Failed to add datasource '{ds_item_name}': {e}")
                continue
            
            # Get the datasource object
            try:
                datasources = data_agent.get_datasources()
                datasource = None
                for ds_obj in datasources:
                    # Match by name — the SDK doesn't expose a direct lookup
                    if hasattr(ds_obj, 'name') and ds_obj.name == ds_item_name:
                        datasource = ds_obj
                        break
                if datasource is None:
                    # Fallback: take the last added datasource
                    datasource = datasources[-1]
            except Exception as e:
                print(f"  ⚠ Cannot retrieve datasource object: {e}")
                continue
            
            # Step 3a: Select tables
            selected_tables = ds_def.get("selected_tables", [])
            for table_name in selected_tables:
                try:
                    datasource.select("dbo", table_name)
                except Exception:
                    # Try without schema prefix
                    try:
                        datasource.select("", table_name)
                    except Exception as e:
                        print(f"    ⚠ Cannot select table '{table_name}': {e}")
            if selected_tables:
                print(f"    ✓ Selected {len(selected_tables)} tables.")
            
            # Step 3b: Set per-datasource instructions
            ds_instructions = ds_def.get("data_instructions", "")
            if ds_instructions:
                try:
                    datasource.update_configuration(instructions=ds_instructions[:15000])
                    print(f"    ✓ Data instructions set.")
                except Exception as e:
                    print(f"    ⚠ Failed to set data instructions: {e}")
            
            # Step 3c: Add example queries (not for Semantic Models)
            example_queries = ds_def.get("example_queries", [])
            if example_queries and ds_item_type != "SemanticModel":
                fewshot_dict = {}
                for eq in example_queries[:MAX_EXAMPLES_PER_DATASOURCE]:
                    q = eq.get("question", "")
                    sql = eq.get("sql", "")
                    if q and sql:
                        fewshot_dict[q] = sql
                
                if fewshot_dict:
                    try:
                        datasource.add_fewshots(fewshot_dict)
                        print(f"    ✓ Added {len(fewshot_dict)} example queries.")
                    except Exception as e:
                        print(f"    ⚠ Failed to add example queries: {e}")
        
        # Step 4: Publish
        try:
            data_agent.publish()
            print(f"  ✓ Agent published!")
            status = "published"
        except Exception as e:
            print(f"  ⚠ Failed to publish agent: {e}")
            status = "created_not_published"
        
        created_agents.append({
            "agent_name": agent_name,
            "status": status,
            "data_sources": len(agent_def.get("data_sources", [])),
            "description": agent_def.get("description", ""),
        })
        
        # Small delay to avoid rate limits
        time.sleep(2)
    
    return created_agents


# Execute
print(f"DRY_RUN = {DRY_RUN}")
created_agents = create_agents_from_plan(agent_plan, DRY_RUN)

---
## Phase 5 — Report & Agent Garden Catalog

Generates a summary report and optionally writes the catalog to a Delta table.

In [ ]:
# Build catalog DataFrame
catalog_records = []
for agent_def, result in zip(agent_plan.get("agents", []), created_agents):
    ds_names = [ds["item_name"] for ds in agent_def.get("data_sources", [])]
    ds_types = [ds["item_type"] for ds in agent_def.get("data_sources", [])]
    all_tables = []
    for ds in agent_def.get("data_sources", []):
        all_tables.extend(ds.get("selected_tables", []))
    
    catalog_records.append({
        "agent_name": result["agent_name"],
        "description": agent_def.get("description", ""),
        "status": result["status"],
        "data_source_count": len(ds_names),
        "data_sources": ", ".join(ds_names),
        "data_source_types": ", ".join(ds_types),
        "tables": ", ".join(all_tables),
        "table_count": len(all_tables),
        "system_instructions_length": len(agent_def.get("system_instructions", "")),
        "total_example_queries": sum(
            len(ds.get("example_queries", [])) for ds in agent_def.get("data_sources", [])
        ),
    })

catalog_df = pd.DataFrame(catalog_records)

print("\n" + "=" * 80)
print("AGENT GARDEN CATALOG — Summary")
print("=" * 80)
display(catalog_df)

print(f"\nTotal agents: {len(catalog_df)}")
print(f"Total data sources used: {catalog_df['data_source_count'].sum()}")
print(f"Total tables mapped: {catalog_df['table_count'].sum()}")
print(f"Total example queries: {catalog_df['total_example_queries'].sum()}")

In [ ]:
# Write catalog to Delta table (optional)
if CATALOG_LAKEHOUSE and not DRY_RUN:
    print(f"Writing catalog to '{CATALOG_LAKEHOUSE}.{CATALOG_TABLE_NAME}'...")
    
    spark_df = spark.createDataFrame(catalog_df)
    spark_df.write.mode("overwrite").saveAsTable(f"{CATALOG_LAKEHOUSE}.{CATALOG_TABLE_NAME}")
    
    print(f"✓ Catalog written to Delta table '{CATALOG_LAKEHOUSE}.{CATALOG_TABLE_NAME}'.")
    print("  You can now build a Power BI report on this table for the Agent Garden UI.")
else:
    if DRY_RUN:
        print("Catalog write skipped (DRY_RUN mode).")
    elif not CATALOG_LAKEHOUSE:
        print("Catalog write skipped (CATALOG_LAKEHOUSE not configured).")

In [ ]:
# Save the full agent plan as JSON artifact for reference
plan_json = json.dumps(agent_plan, indent=2, ensure_ascii=False)
print("Full agent plan JSON (for reference):")
print(plan_json[:5000])
if len(plan_json) > 5000:
    print(f"\n... ({len(plan_json):,} total characters. Full plan available in 'agent_plan' variable.)")

---
## Done!

### Next Steps:

1. **Review dry-run output** above — check that agent groupings and instructions make sense
2. **Set `DRY_RUN = False`** in Cell 0 and re-run from Phase 4 to actually create the agents
3. **Test agents** by navigating to them in the Fabric workspace and asking questions
4. **Share agents** via the Copilot in Power BI integration or published URLs
5. **Build Agent Garden UI** by creating a Power BI report on the `agent_garden_catalog` Delta table

### Troubleshooting:

- **"Cannot list items"**: Ensure you have Viewer (or higher) role on the target workspaces
- **"Cannot profile table"**: The Spark context must have access to the lakehouse data (may need shortcuts)
- **"Failed to add datasource"**: The item must be accessible via the OneLake catalog from the notebook's workspace
- **Azure OpenAI errors**: Verify `AOAI_ENDPOINT` and `AOAI_DEPLOYMENT` are correct and that your token has access
- **Max 5 data sources**: The notebook auto-splits agents that exceed this limit